# CELLxGENE — Download All Visium Datasets

This notebook uses the [CZ CELLxGENE Discover API](https://api.cellxgene.cziscience.com) to:

1. Query all collections and find datasets with the **Visium Spatial Gene Expression** assay.
2. Build a metadata catalog (tissue, organism, cell count, collection, download URL).
3. Optionally download the `.h5ad` files to a local directory.

No extra packages needed beyond `requests` and `pandas`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

API_BASE = "https://api.cellxgene.cziscience.com/curation/v1"
DOWNLOAD_DIR = Path("/Volumes/processing/cellxgene/cellxgene_visium")

# Set to True to actually download .h5ad files (can be many GB!)
DO_DOWNLOAD = True

# Session with retry + longer timeouts for the slow CELLxGENE API
session = requests.Session()
session.mount(
    "https://",
    HTTPAdapter(max_retries=Retry(total=3, backoff_factor=2, status_forcelist=[502, 503, 504])),
)
API_TIMEOUT = 180  # seconds

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 60)

## 1. Fetch all collections and find Visium datasets

The lightweight `/collections` endpoint returns assay labels per dataset. We filter for anything matching **Visium**.

In [2]:
resp = session.get(f"{API_BASE}/collections", timeout=API_TIMEOUT)
resp.raise_for_status()
collections = resp.json()
print(f"Total collections: {len(collections)}")

# Collect dataset_ids and collection_ids for Visium datasets
visium_hits = []
for c in collections:
    for d in c.get("datasets", []):
        assay_labels = [a.get("label", "") for a in d.get("assay", [])]
        if any("visium" in label.lower() for label in assay_labels):
            visium_hits.append({
                "collection_id": c["collection_id"],
                "collection_name": c["name"],
                "dataset_id": d["dataset_id"],
                "dataset_version_id": d["dataset_version_id"],
                "assay": ", ".join(assay_labels),
                "tissue": ", ".join(t.get("label", "") for t in d.get("tissue", [])),
                "organism": ", ".join(o.get("label", "") for o in d.get("organism", [])),
            })

print(f"Visium datasets found: {len(visium_hits)}")
print(f"Across {len(set(h['collection_id'] for h in visium_hits))} collections")

Total collections: 362
Visium datasets found: 338
Across 22 collections


## 2. Fetch full metadata and download URLs

The `/collections/{id}` endpoint includes `assets` with the `.h5ad` download link, plus cell counts, titles, etc.

In [3]:
from time import sleep

# Fetch full metadata per collection (deduplicated — many datasets share a collection)
collection_ids = sorted(set(h["collection_id"] for h in visium_hits))
full_collections = {}
for i, cid in enumerate(collection_ids):
    if cid not in full_collections:
        r = session.get(f"{API_BASE}/collections/{cid}", timeout=API_TIMEOUT)
        r.raise_for_status()
        full_collections[cid] = r.json()
        if (i + 1) % 10 == 0:
            print(f"  fetched {i + 1}/{len(collection_ids)} collections...")
        sleep(0.2)  # be polite

print(f"Fetched {len(full_collections)} collections")

  fetched 10/22 collections...
  fetched 20/22 collections...
Fetched 22 collections


In [4]:
# Build the full catalog by matching dataset_ids to their full metadata
visium_dataset_ids = {h["dataset_id"] for h in visium_hits}

catalog = []
for cid, coll in full_collections.items():
    # Collection-level provenance (= the original publication)
    doi = coll.get("doi", "")
    collection_url = coll.get("collection_url", "")
    pub_meta = coll.get("publisher_metadata") or {}
    journal = pub_meta.get("journal", "")
    # GEO / raw data links
    geo_links = ", ".join(
        link["link_url"] for link in coll.get("links", [])
        if link.get("link_type") == "RAW_DATA"
    )

    for d in coll.get("datasets", []):
        if d["dataset_id"] not in visium_dataset_ids:
            continue

        h5ad_assets = [a for a in d.get("assets", []) if a.get("filetype") == "H5AD"]
        h5ad_url = h5ad_assets[0]["url"] if h5ad_assets else None
        h5ad_size_mb = round(h5ad_assets[0]["filesize"] / 1e6, 1) if h5ad_assets else None

        catalog.append({
            "dataset_id": d["dataset_id"],
            "collection_id": cid,
            "title": d.get("title", ""),
            "collection": coll["name"],
            "doi": doi,
            "journal": journal,
            "collection_url": collection_url,
            "raw_data_links": geo_links,
            "organism": ", ".join(o.get("label", "") for o in d.get("organism", [])),
            "tissue": ", ".join(t.get("label", "") for t in d.get("tissue", [])),
            "disease": ", ".join(dis.get("label", "") for dis in d.get("disease", [])),
            "cell_count": d.get("cell_count"),
            "assay": ", ".join(a.get("label", "") for a in d.get("assay", [])),
            "h5ad_url": h5ad_url,
            "h5ad_size_mb": h5ad_size_mb,
            "explorer_url": d.get("explorer_url"),
        })

df = pd.DataFrame(catalog)
print(f"Catalog: {len(df)} Visium datasets")
print(f"Total download size: {df['h5ad_size_mb'].sum() / 1000:.1f} GB")
print(f"\nOrganisms:  {df['organism'].value_counts().to_dict()}")
print(f"Top tissues: {df['tissue'].value_counts().head(10).to_dict()}")
df.head(10)

Catalog: 338 Visium datasets
Total download size: 91.6 GB

Organisms:  {'Homo sapiens': 307, 'Mus musculus': 31}
Top tissues: {'thymus': 37, 'heart left ventricle': 34, 'breast': 32, 'labial gland': 24, 'lung': 20, 'kidney': 18, 'liver': 18, 'colon': 14, 'sinoatrial node': 13, 'bronchus': 10}


,dataset_id,collection_id,title,collection,doi,journal,collection_url,raw_data_links,organism,tissue,disease,cell_count,assay,h5ad_url,h5ad_size_mb,explorer_url
0,fca00c4c-0b8a-4836-ab7b-6620123d2d4e,0c8a364b-97b5-4cc8-a593-23c38c6f0ac5,Visium spatial data from liver of primary sclerosing cho...,Single-cell and spatial transcriptomics characterisation...,10.1016/j.jhep.2023.12.023,Journal of Hepatology,https://cellxgene.cziscience.com/collections/0c8a364b-97...,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE24...,Homo sapiens,caudate lobe of liver,primary sclerosing cholangitis,4992,Visium Spatial Gene Expression V1,https://datasets.cellxgene.cziscience.com/0cc59004-2b35-...,53.3,https://cellxgene.cziscience.com/e/fca00c4c-0b8a-4836-ab...
1,a0805e92-3d1f-4866-af5c-5b903c6d910b,0c8a364b-97b5-4cc8-a593-23c38c6f0ac5,Visium spatial data from liver of primary sclerosing cho...,Single-cell and spatial transcriptomics characterisation...,10.1016/j.jhep.2023.12.023,Journal of Hepatology,https://cellxgene.cziscience.com/collections/0c8a364b-97...,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE24...,Homo sapiens,caudate lobe of liver,primary sclerosing cholangitis,4992,Visium Spatial Gene Expression V1,https://datasets.cellxgene.cziscience.com/721bfc1c-f77f-...,44.6,https://cellxgene.cziscience.com/e/a0805e92-3d1f-4866-af...
2,7f7e36ee-7432-4b70-b894-25d9989d43e8,0c8a364b-97b5-4cc8-a593-23c38c6f0ac5,Visium spatial data from human healthy liver donor; samp...,Single-cell and spatial transcriptomics characterisation...,10.1016/j.jhep.2023.12.023,Journal of Hepatology,https://cellxgene.cziscience.com/collections/0c8a364b-97...,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE24...,Homo sapiens,caudate lobe of liver,normal,4992,Visium Spatial Gene Expression V1,https://datasets.cellxgene.cziscience.com/9c731db0-9f44-...,1108.6,https://cellxgene.cziscience.com/e/7f7e36ee-7432-4b70-b8...
3,7ca2e329-d91b-4e36-b4cc-663447ae437e,0c8a364b-97b5-4cc8-a593-23c38c6f0ac5,Visium spatial data from human healthy liver donor; samp...,Single-cell and spatial transcriptomics characterisation...,10.1016/j.jhep.2023.12.023,Journal of Hepatology,https://cellxgene.cziscience.com/collections/0c8a364b-97...,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE24...,Homo sapiens,caudate lobe of liver,normal,4992,Visium Spatial Gene Expression V1,https://datasets.cellxgene.cziscience.com/44453de7-1d66-...,1096.9,https://cellxgene.cziscience.com/e/7ca2e329-d91b-4e36-b4...
4,7b814d77-810f-49e5-ae73-c3b644ea197f,0c8a364b-97b5-4cc8-a593-23c38c6f0ac5,Visium spatial data from human healthy liver donor; samp...,Single-cell and spatial transcriptomics characterisation...,10.1016/j.jhep.2023.12.023,Journal of Hepatology,https://cellxgene.cziscience.com/collections/0c8a364b-97...,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE24...,Homo sapiens,caudate lobe of liver,normal,4992,Visium Spatial Gene Expression V1,https://datasets.cellxgene.cziscience.com/1fa9f77e-b79e-...,1080.7,https://cellxgene.cziscience.com/e/7b814d77-810f-49e5-ae...
5,7a90a1ce-4e66-44d0-b273-7c672de53b17,0c8a364b-97b5-4cc8-a593-23c38c6f0ac5,Visium spatial data from liver of primary sclerosing cho...,Single-cell and spatial transcriptomics characterisation...,10.1016/j.jhep.2023.12.023,Journal of Hepatology,https://cellxgene.cziscience.com/collections/0c8a364b-97...,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE24...,Homo sapiens,caudate lobe of liver,primary sclerosing cholangitis,4992,Visium Spatial Gene Expression V1,https://datasets.cellxgene.cziscience.com/6da751d4-63af-...,52.3,https://cellxgene.cziscience.com/e/7a90a1ce-4e66-44d0-b2...
6,6d222287-cf5b-4eb5-86e3-c4e71adab844,0c8a364b-97b5-4cc8-a593-23c38c6f0ac5,Visium spatial data from human healthy liver donor; samp...,Single-cell and spatial transcriptomics characterisation...,10.1016/j.jhep.2023.12.023,Journal of Hepatology,https://cellxgene.cziscience.com/collections/0c8a364b-97...,https://www.ncbi.nlm.nih.gov/g

## 3. Filter (optional)

Narrow down the catalog before downloading. Adjust the filter below to select specific tissues, organisms, or size limits.

In [5]:
# Example filters — comment out or adjust as needed
df_filtered = df.copy()

# Uncomment to filter by organism:
# df_filtered = df_filtered[df_filtered["organism"].str.contains("Homo sapiens", case=False)]

# Uncomment to filter by tissue:
# df_filtered = df_filtered[df_filtered["tissue"].str.contains("brain", case=False)]

# Uncomment to limit file size (in MB):
# df_filtered = df_filtered[df_filtered["h5ad_size_mb"] < 500]

print(f"Selected: {len(df_filtered)} datasets ({df_filtered['h5ad_size_mb'].sum() / 1000:.1f} GB)")
df_filtered[["title", "organism", "tissue", "cell_count", "h5ad_size_mb"]].head(20)

Selected: 338 datasets (91.6 GB)


,title,organism,tissue,cell_count,h5ad_size_mb
0,Visium spatial data from liver of primary sclerosing cho...,Homo sapiens,caudate lobe of liver,4992,53.3
1,Visium spatial data from liver of primary sclerosing cho...,Homo sapiens,caudate lobe of liver,4992,44.6
2,Visium spatial data from human healthy liver donor; samp...,Homo sapiens,caudate lobe of liver,4992,1108.6
3,Visium spatial data from human healthy liver donor; samp...,Homo sapiens,caudate lobe of liver,4992,1096.9
4,Visium spatial data from human healthy liver donor; samp...,Homo sapiens,caudate lobe of liver,4992,1080.7
5,Visium spatial data from liver of primary sclerosing cho...,Homo sapiens,caudate lobe of liver,4992,52.3
6,Visium spatial data from human healthy liver donor; samp...,Homo sapiens,caudate lobe of liver,4992,1172.4
7,Visium spatial data from liver of primary sclerosing cho...,Homo sapiens,caudate lobe of liver,4992,61.8
8,spRNA-seq for GZMK+CD8+ T cells Target A Specific Acinar...,Homo sapiens,labial gland,4992,31.9
9,spRNA-seq for GZMK+CD8+ T cells Target A Specific Acinar...,Homo sapiens,labial gland,4992,36.3


## 4. Download

Downloads each `.h5ad` into a subfolder per collection (publication). Folder names are derived from the collection name. Skips files that already exist. Gated by `DO_DOWNLOAD = True`.

In [6]:
import re


def _sanitize_folder_name(name: str, max_len: int = 80) -> str:
    """Turn a collection name into a safe, readable folder name."""
    name = name.strip()
    name = re.sub(r"[^\w\s\-]", "", name)       # drop special chars
    name = re.sub(r"\s+", "_", name)             # spaces → underscores
    return name[:max_len].rstrip("_")


def download_h5ad(url: str, dest: Path) -> None:
    """Stream-download a file with progress."""
    with session.get(url, stream=True, timeout=300) as r:
        r.raise_for_status()
        downloaded = 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
                downloaded += len(chunk)
        print(f"    saved {downloaded / 1e6:.1f} MB")


def _write_readme(folder: Path, rows: "pd.DataFrame") -> None:
    """Write a README.txt with publication provenance into the collection folder."""
    row = rows.iloc[0]
    lines = [
        f"Collection: {row['collection']}",
        f"DOI: {row['doi']}",
        f"Journal: {row['journal']}",
        f"CELLxGENE: {row['collection_url']}",
        f"Raw data: {row['raw_data_links']}",
        "",
        f"Datasets ({len(rows)}):",
    ]
    for _, r in rows.iterrows():
        lines.append(f"  - {r['dataset_id']}.h5ad  {r['title']}")
    (folder / "README.txt").write_text("\n".join(lines) + "\n")


# Build folder mapping
collection_folders = {}
for cid, name in df_filtered[["collection_id", "collection"]].drop_duplicates().values:
    collection_folders[cid] = _sanitize_folder_name(name)

# Check what already exists
already_downloaded = 0
to_download = 0
skip_no_url = 0
for _, row in df_filtered.iterrows():
    if row["h5ad_url"] is None:
        skip_no_url += 1
    else:
        dest = DOWNLOAD_DIR / collection_folders[row["collection_id"]] / f"{row['dataset_id']}.h5ad"
        if dest.exists():
            already_downloaded += 1
        else:
            to_download += 1

print(f"Already downloaded: {already_downloaded}")
print(f"To download:       {to_download}")
if skip_no_url:
    print(f"No URL available:  {skip_no_url}")
print()

if DO_DOWNLOAD:
    DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

    for i, row in df_filtered.iterrows():
        url = row["h5ad_url"]
        if url is None:
            continue

        folder = DOWNLOAD_DIR / collection_folders[row["collection_id"]]
        folder.mkdir(parents=True, exist_ok=True)
        dest = folder / f"{row['dataset_id']}.h5ad"

        if dest.exists():
            continue

        print(f"  [{i}] {collection_folders[row['collection_id']]}/")
        print(f"        {row['title'][:60]}  ({row['h5ad_size_mb']} MB)")
        download_h5ad(url, dest)

    # Write a README.txt per collection folder with provenance
    for cid, folder_name in collection_folders.items():
        folder = DOWNLOAD_DIR / folder_name
        if folder.exists():
            coll_rows = df_filtered[df_filtered["collection_id"] == cid]
            _write_readme(folder, coll_rows)

    print(f"\nDone. Folder structure:")
    for folder in sorted(DOWNLOAD_DIR.iterdir()):
        if folder.is_dir():
            h5ads = list(folder.glob("*.h5ad"))
            size = sum(p.stat().st_size for p in h5ads) / 1e6
            print(f"  {folder.name}/  ({len(h5ads)} files, {size:.0f} MB)")
else:
    print("DO_DOWNLOAD is False — set to True to start downloading.")
    print(f"Planned download dir: {DOWNLOAD_DIR}\n")
    print(f"Collection folders ({len(collection_folders)}):")
    for cid, folder in sorted(collection_folders.items(), key=lambda x: x[1]):
        n = len(df_filtered[df_filtered["collection_id"] == cid])
        doi = df_filtered[df_filtered["collection_id"] == cid]["doi"].iloc[0]
        print(f"  {folder}/  ({n} datasets)  DOI: {doi}")


Already downloaded: 78
To download:       260

  [4] Single-cell_and_spatial_transcriptomics_characterisation_of_the_immunological_la/
        Visium spatial data from human healthy liver donor; sample C  (1080.7 MB)
    saved 1080.7 MB
  [5] Single-cell_and_spatial_transcriptomics_characterisation_of_the_immunological_la/
        Visium spatial data from liver of primary sclerosing cholang  (52.3 MB)
    saved 52.3 MB
  [6] Single-cell_and_spatial_transcriptomics_characterisation_of_the_immunological_la/
        Visium spatial data from human healthy liver donor; sample C  (1172.4 MB)
    saved 1172.4 MB
  [7] Single-cell_and_spatial_transcriptomics_characterisation_of_the_immunological_la/
        Visium spatial data from liver of primary sclerosing cholang  (61.8 MB)
    saved 61.8 MB
  [82] Spatially_resolved_multiomics_of_human_cardiac_niches/
        Visium spatial - HCAHeartST9341982 (OCT)  (224.7 MB)
    saved 224.7 MB
  [83] Spatially_resolved_multiomics_of_human_cardiac_niche

## 5. Save catalog to CSV

In [7]:
catalog_path = DOWNLOAD_DIR / "visium_catalog.csv"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(catalog_path, index=False)
print(f"Saved catalog: {catalog_path}")
print(f"{len(df)} datasets")

Saved catalog: /Volumes/processing/cellxgene/cellxgene_visium/visium_catalog.csv
338 datasets
